# 향수 추천 시스템 모델 구현

| 모델 | 입력 데이터 | 학습·추천 방식 | 반환값 |
|---|---|---|---|
| 콘텐츠 기반 | Fragrantica의 `content_text` + `brand` | TF-IDF + 코사인 유사도 | 유사도와 추천 향수 |
| 메모리 기반 협업 | Atrafshan의 사용자별 `scent_rating` | Item-based KNN | 예상 평점과 추천 향수 |

모델 구현 및 동작 확인용 노트북. 아래 전체 평점 학습 셀은 추천 동작 예시이며 성능 평가 결과가 아님. 

모델 구현은 모두 이 노트북에 포함. CSV가 준비되어 있다면 이 파일과 `requirements.txt`만으로 실행 가능.

## 1. 실행 환경

Python 3.11 이상과 Jupyter 커널 사용. 최초 1회 아래 설치 명령의 주석을 해제해 실행.

기존 CSV 5개가 있는 폴더를 `DATA_DIR`로 지정하고 셀을 위에서부터 실행. 기본 경로는 노트북과 같은 위치의 `data/` 폴더.

In [2]:
# 최초 1회만 주석을 해제하여 실행하세요.
# %pip install -r requirements.txt
# requirements.txt가 없는 경우:
# %pip install "numpy>=1.26,<3" "pandas>=2.2,<3" "scipy>=1.12,<2" "scikit-learn>=1.5,<2" ipykernel

import sys
import sklearn
import scipy
from IPython.display import display

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__, "/ SciPy:", scipy.__version__)

Python: 3.13.15
scikit-learn: 1.9.1 / SciPy: 1.18.1


## 2. 데이터 로딩 및 입력 검증

중복 ID, 1~10점 범위, 사용자–향수 중복, 자동 매칭 상태, 연결된 평점의 일치 여부만 확인

| 파일 | 역할 |
|---|---|
| `perfumes_content.csv` | 콘텐츠 기반 모델의 향수 특징 |
| `user_ratings.csv` | 전체 유효 평점으로 KNN 학습 |
| `user_ratings_linked.csv` | Atrafshan 사용자의 선호를 Fragrantica ID로 전달 |
| `perfume_mapping.csv` | `matched_exact` ID 연결만 사용 |
| `atrafshan_perfumes.csv` | KNN 추천의 이름·브랜드 표시 |

- `atrafshan_reviews_preserved.csv`는 원본 보존용
- 결측·불명확한 0점이 포함되므로 학습에 사용하지 않음

In [ ]:
#TF-IDF content filtering and explicit-rating item-based KNN.
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.base import BaseEstimator
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sklearn.utils.validation import check_is_fitted

# 입력 데이터에 필요한 컬럼이 모두 있는지 확인
def _require_columns(frame, columns):
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f"Missing columns: {missing}")


# 사용자·향수 ID의 결측, 빈 값, 앞뒤 공백을 검사
def _check_ids(frame, columns):
    # ID를 문자열로 확인하되 서로 다른 사이트의 ID를 임의로 합치지는 않음
    for column in columns:
        if frame[column].isna().any():
            raise ValueError(f"Missing identifier in {column}")
        values = frame[column].astype(str)
        if values.str.strip().eq("").any() or values.ne(values.str.strip()).any():
            raise ValueError(f"Empty or whitespace-padded identifier in {column}")


# 이웃 수나 추천 개수가 1 이상의 정수인지 확인
def _positive_int(value, name):
    if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < 1:
        raise ValueError(f"{name} must be a positive integer")


# 평점 데이터의 범위와 중복을 검사
def _validate_ratings(ratings):
    columns = ["user_id", "perfume_id", "scent_rating"]
    _require_columns(ratings, columns)
    result = ratings[columns].copy()
    _check_ids(result, ["user_id", "perfume_id"])
    result[["user_id", "perfume_id"]] = result[["user_id", "perfume_id"]].astype(str)
    # 문자열 평점을 숫자로 바꾸고 실제 평가 범위만 허용
    result["scent_rating"] = pd.to_numeric(result["scent_rating"], errors="raise")
    if not result.scent_rating.between(1, 10).all():
        raise ValueError("Only observed scent ratings in [1, 10] are allowed")
    if result.duplicated(["user_id", "perfume_id"]).any():
        raise ValueError("Duplicate user-item pairs: resolve them before fit()")
    return result


# CSV 파일 5개를 읽고 평점·매핑·연결 파일이 서로 일치하는지 확인
def load_data(data_dir):
    data_dir = Path(data_dir)
    names = {"content": "perfumes_content.csv", "ratings": "user_ratings.csv",
             "linked": "user_ratings_linked.csv", "mapping": "perfume_mapping.csv",
             "atrafshan": "atrafshan_perfumes.csv"}
    # 필수 파일이 없으면 경로를 수정하도록 안내
    absent = [name for name in names.values() if not (data_dir / name).is_file()]
    if absent:
        raise FileNotFoundError(f"Set DATA_DIR to the CSV folder. Missing: {absent}")
    # ID의 접두사와 원래 값을 보존하도록 CSV를 문자열로 읽고 결측값은 빈 문자열로 처리
    data = {key: pd.read_csv(data_dir / name, dtype=str, keep_default_na=False)
            for key, name in names.items()}
    # 학습용 평점과 연결된 평점에 같은 유효성 검사를 적용
    for key in ["ratings", "linked"]:
        data[key]["scent_rating"] = _validate_ratings(data[key])["scent_rating"]
    _require_columns(data["content"], ["perfume_id", "content_text", "brand"])
    _require_columns(data["atrafshan"], ["perfume_id", "perfume_name", "brand"])
    _require_columns(data["mapping"], ["atrafshan_id", "fragrantica_id", "match_status"])
    _require_columns(data["linked"], ["fragrantica_id"])
    for key in ["content", "atrafshan"]:
        _check_ids(data[key], ["perfume_id"])
        if data[key].perfume_id.duplicated().any():
            raise ValueError(f"Duplicate catalog IDs in {key}")
    mapping = data["mapping"]
    _check_ids(mapping, ["atrafshan_id"])
    if mapping.atrafshan_id.duplicated().any():
        raise ValueError("Duplicate Atrafshan IDs in mapping")
    # 확정된 자동 매칭만 사용하고 미연결된 행은 연결하지 않음
    exact = mapping.loc[mapping.match_status.eq("matched_exact")]
    _check_ids(exact, ["fragrantica_id"])
    if exact.fragrantica_id.duplicated().any():
        raise ValueError("Multiple Atrafshan IDs map to one Fragrantica ID")
    if not exact.fragrantica_id.isin(data["content"].perfume_id).all():
        raise ValueError("Mapping refers to missing Fragrantica IDs")
    if not mapping.atrafshan_id.isin(data["atrafshan"].perfume_id).all():
        raise ValueError("Mapping refers to missing Atrafshan IDs")
    if not data["ratings"].perfume_id.isin(data["atrafshan"].perfume_id).all():
        raise ValueError("Ratings refer to missing Atrafshan IDs")
    # 전체 평점과 매핑으로 예상 연결 결과를 만들어 linked 파일과 비교
    expected = data["ratings"].merge(exact[["atrafshan_id", "fragrantica_id"]],
        left_on="perfume_id", right_on="atrafshan_id", validate="many_to_one")
    columns = ["user_id", "perfume_id", "scent_rating", "fragrantica_id"]
    actual = data["linked"][columns].sort_values(columns[:2]).reset_index(drop=True)
    expected = expected[columns].sort_values(columns[:2]).reset_index(drop=True)
    if not actual.equals(expected):
        raise ValueError("Linked ratings disagree with ratings + matched_exact mapping")
    return data


In [ ]:
DATA_DIR = Path("data")  # csv 경로가 달라질 시 이 부분에서 경로를 수정

data = load_data(DATA_DIR)
perfumes = data["content"]
ratings = data["ratings"]
linked_ratings = data["linked"]
mapping = data["mapping"]
atrafshan = data["atrafshan"]

display(pd.DataFrame({
    "File": ["perfumes_content.csv", "user_ratings.csv", "user_ratings_linked.csv",
           "perfume_mapping.csv", "atrafshan_perfumes.csv"],
    "행 수": [len(perfumes), len(ratings), len(linked_ratings), len(mapping), len(atrafshan)]
}))
n_users, n_items = ratings.user_id.nunique(), ratings.perfume_id.nunique()
print(f"KNN: 사용자 {n_users:,}명 / 향수 {n_items:,}개 / 평점 {len(ratings):,}건")
print(f"관측 평점 비율: {len(ratings) / (n_users * n_items):.4%}")
print("사용자당 평점 수 중앙값:", ratings.groupby("user_id").size().median())
display(mapping.match_status.value_counts().rename_axis("match_status").to_frame("향수 수"))

,File,행 수
0,perfumes_content.csv,24063
1,user_ratings.csv,5565
2,user_ratings_linked.csv,3530
3,perfume_mapping.csv,1369
4,atrafshan_perfumes.csv,1369


KNN: 사용자 1,747명 / 향수 1,050개 / 평점 5,565건
관측 평점 비율: 0.3034%
사용자당 평점 수 중앙값: 1.0


,향수 수
match_status,
matched_exact,882
unmatched,457
review_required,30


## 3. 콘텐츠 기반 모델: TF-IDF + 코사인 유사도

`content_text`의 `note_pink_pepper` 같은 토큰은 공백으로만 분리. 노트와 어코드의 구분, 여러 단어로 이루어진 향료를 그대로 유지. 설명, 평균 평점, 평가 수는 특징에 포함하지 않음.

**브랜드 + 향 특징**을 반영하기 위해 두 특징 블록을 별도로 구성. 전처리된 `content_text`를 수정하지 않고, 브랜드는 별도의 TF-IDF로 표현. 브랜드 하나가 토큰 하나이므로 정규화 후 브랜드 일치 여부를 나타내는 특징으로 사용.

$$x_i = \operatorname{normalize}\left[\sqrt{1-\beta}\,t_i\ ;\ \sqrt{\beta}\,b_i\right]$$

- $t_i$: 노트·어코드의 L2 정규화 TF-IDF 벡터
- $b_i$: 브랜드의 L2 정규화 TF-IDF 벡터
- 기본 $\beta=0.2$: 두 블록이 있는 향수끼리의 유사도는 향 특징 80%, 동일 브랜드 20%를 반영
- `brand_weight=0.0`: 전처리 설명에 있는 **노트·어코드만 사용하는 방식**
- 0.2는 구현을 위한 초기 설정

여러 선호 향수를 입력하면 벡터의 가중 평균으로 사용자 프로필을 만들고 후보와 코사인 유사도를 계산합니다. 사용자 평점 이력을 사용할 때는 기본 7점 이상을 선호로 정하고 실제 평점을 프로필 가중치로 사용합니다. 이미 평가한 향수는 낮은 점수를 준 향수까지 모두 제외합니다.


In [ ]:
# 향 특징과 브랜드의 TF-IDF 벡터로 유사한 향수를 추천합니다.
class ContentBasedRecommender(BaseEstimator):

    # 브랜드 가중치를 설정. 0이면 향 특징만 사용.
    def __init__(self, brand_weight=0.2):
        if not np.isfinite(brand_weight) or not 0 <= brand_weight < 1:
            raise ValueError("brand_weight must be in [0, 1)")
        self.brand_weight = float(brand_weight)

    # 향수별 향의 특징, 브랜드를 TF-IDF로 변환하고 검색용 벡터 생성.
    def fit(self, perfumes):
        _require_columns(perfumes, ["perfume_id", "brand", "content_text"])
        _check_ids(perfumes, ["perfume_id"])
        if perfumes.perfume_id.astype(str).duplicated().any():
            raise ValueError("Duplicate Fragrantica perfume IDs")
        # 향 특징이 비어 있거나 사용 불가인 향수는 추천 대상에서 제외.
        eligible = perfumes.content_text.fillna("").astype(str).str.strip().ne("")
        if "content_eligible" in perfumes:
            flag = perfumes.content_eligible.astype(str).str.lower()
            if not flag.isin(["true", "false"]).all():
                raise ValueError("content_eligible must contain True/False")
            eligible &= flag.eq("true")
        self.catalog_ = perfumes.loc[eligible].reset_index(drop=True).copy()
        self.catalog_["perfume_id"] = self.catalog_.perfume_id.astype(str)
        self.excluded_ids_ = perfumes.loc[~eligible, "perfume_id"].astype(str).tolist()
        if self.catalog_.empty:
            raise ValueError("No eligible perfume has scent features")
        self.item_index_ = pd.Index(self.catalog_.perfume_id, name="perfume_id")
        # 공백만 분리해 note_pink_pepper와 같은 다중 단어 토큰을 유지.
        options = dict(tokenizer=str.split, token_pattern=None, lowercase=False,
                       norm="l2", dtype=np.float32)
        self.note_vectorizer_ = TfidfVectorizer(**options)
        notes = self.note_vectorizer_.fit_transform(self.catalog_.content_text.astype(str))
        # 코사인 유사도에서 향 특징 80%·브랜드 20%가 되도록 제곱근 가중치를 설정
        blocks = [notes * np.sqrt(1 - self.brand_weight)]
        self.brand_vectorizer_ = None
        if self.brand_weight > 0:
            # 브랜드 전체를 한 토큰으로 만들어 같은 브랜드인지 확인
            column = "brand_normalized" if "brand_normalized" in self.catalog_ else "brand"
            brands = self.catalog_[column].fillna("").astype(str).str.casefold().str.strip()
            tokens = brands.str.replace(r"\s+", "_", regex=True)
            tokens = tokens.where(tokens.eq(""), "brand_" + tokens)
            if tokens.ne("").any():
                self.brand_vectorizer_ = TfidfVectorizer(**options)
                blocks.append(self.brand_vectorizer_.fit_transform(tokens) * np.sqrt(self.brand_weight))
        # 두 특징 블록을 합친 뒤 길이를 1로 맞춰 코사인 계산에 사용
        self.tfidf_matrix_ = normalize(hstack(blocks, format="csr"), norm="l2")
        self.display_columns_ = [c for c in ["perfume_id", "perfume_name", "brand", "gender", "url"]
                                 if c in self.catalog_]
        return self

    # 선호 향수로 만든 프로필과 각 후보의 코사인 유사도를 반환
    def score_for_likes(self, liked_ids, weights=None, candidate_ids=None):
        check_is_fitted(self, ["tfidf_matrix_"])
        liked_ids = [str(item) for item in liked_ids]
        if not liked_ids or len(liked_ids) != len(set(liked_ids)):
            raise ValueError("Supply one or more distinct liked perfume IDs")
        # 입력 향수 ID를 행렬의 행 번호로 변환
        indices = self.item_index_.get_indexer(liked_ids)
        if (indices < 0).any():
            raise KeyError(f"Unknown/ineligible liked IDs: {np.array(liked_ids)[indices < 0].tolist()}")
        weights = np.ones(len(indices)) if weights is None else np.asarray(weights, dtype=float)
        if weights.shape != (len(indices),) or not np.isfinite(weights).all() or (weights <= 0).any():
            raise ValueError("weights must be finite positive values, one per liked perfume")
        # 선호 향수 벡터를 입력 가중치에 따라 평균을 계산해 프로필 생성
        profile = csr_matrix((weights / weights.sum()).reshape(1, -1)) @ self.tfidf_matrix_[indices]
        # 후보를 지정하면 그 범위에서만 점수를 계산
        candidates = self.item_index_ if candidate_ids is None else pd.Index(list(map(str, candidate_ids)))
        candidate_indices = self.item_index_.get_indexer(candidates)
        if (candidate_indices < 0).any() or candidates.has_duplicates:
            raise ValueError("Candidate IDs must be distinct eligible Fragrantica IDs")
        if len(candidates) == 0:
            return pd.Series([], index=candidates, dtype=float, name="score")
        # 전체가 아닌 현재 프로필과 후보 사이의 유사도만 계산
        scores = cosine_similarity(profile, self.tfidf_matrix_[candidate_indices]).ravel()
        return pd.Series(np.clip(scores, 0, 1), index=candidates, name="score")

    # 선호·제외 향수를 빼고 유사도가 높은 향수를 추천
    def recommend_for_likes(self, liked_ids, top_n=10, weights=None,
                            exclude_ids=(), candidate_ids=None):
        _positive_int(top_n, "top_n")
        liked_ids = list(map(str, liked_ids))
        scores = self.score_for_likes(liked_ids, weights, candidate_ids)
        # 선호 향수와 이미 평가한 향수는 추천 결과에서 제외
        excluded = set(liked_ids) | set(map(str, exclude_ids))
        scores = scores.loc[~scores.index.isin(excluded) & scores.gt(1e-8)]
        # 유사도 내림차순으로 정렬하고 동점은 ID 순서로 고정
        ranked = scores.rename_axis("perfume_id").reset_index().sort_values(
            ["score", "perfume_id"], ascending=[False, True], kind="stable").head(top_n)
        return ranked.merge(self.catalog_[self.display_columns_], on="perfume_id", validate="one_to_one")

    # 향수 하나를 기준으로 비슷한 향수를 추천
    def recommend_similar(self, perfume_id, top_n=10):
        return self.recommend_for_likes([perfume_id], top_n=top_n)

    # 입력받은 사용자의 고평점 이력으로 추천하고 평가한 향수는 제외
    def recommend_for_user(self, user_id, train_linked_ratings, top_n=10,
                           like_threshold=7, candidate_ids=None):
        check_is_fitted(self, ["tfidf_matrix_"])
        _positive_int(top_n, "top_n")
        _require_columns(train_linked_ratings, ["user_id", "fragrantica_id", "scent_rating"])
        if not np.isfinite(like_threshold) or not 1 <= like_threshold <= 10:
            raise ValueError("like_threshold must be in [1, 10]")
        # 이 함수에 전달된 이력에서 해당 사용자의 평점만 가져옵니다.
        history = train_linked_ratings.loc[train_linked_ratings.user_id.astype(str).eq(str(user_id))].copy()
        _check_ids(history, ["fragrantica_id"])
        history["scent_rating"] = pd.to_numeric(history.scent_rating, errors="raise")
        if not history.scent_rating.between(1, 10).all():
            raise ValueError("Invalid scent rating in user history")
        if history.fragrantica_id.duplicated().any():
            raise ValueError("Duplicate Fragrantica perfume in user history")
        # like_threshold(기본 값은 7점) 이상의 경우만 선호 프로필에 추가
        liked = history.loc[history.scent_rating.ge(like_threshold)
                            & history.fragrantica_id.isin(self.item_index_)]
        # 사용 가능한 선호 이력이 없으면 임의 추천 대신 빈 결과를 반환
        if liked.empty:
            return pd.DataFrame(columns=["perfume_id", "score"] + self.display_columns_[1:])
        return self.recommend_for_likes(liked.fragrantica_id, top_n, weights=liked.scent_rating,
            exclude_ids=history.fragrantica_id, candidate_ids=candidate_ids)


In [6]:
# 전체 Fragrantica로 TF-IDF를 구성합니다.
content_model = ContentBasedRecommender(brand_weight=0.2).fit(perfumes)
print("TF-IDF 행렬:", content_model.tfidf_matrix_.shape)
print("향 특징이 없어 제외된 향수:", len(content_model.excluded_ids_))

exact_mapping = mapping.loc[mapping.match_status.eq("matched_exact")].copy()
seed = exact_mapping.loc[
    exact_mapping.fragrantica_id.isin(content_model.item_index_)
].iloc[0]
SEED_FR_ID = seed.fragrantica_id
SEED_ATR_ID = seed.atrafshan_id
print("기준 향수:", seed.atrafshan_name, "/", SEED_ATR_ID, "/", SEED_FR_ID)
content_similar = content_model.recommend_similar(SEED_FR_ID, top_n=10)
display(content_similar.drop(columns="url", errors="ignore"))

TF-IDF 행렬: (24063, 2812)
향 특징이 없어 제외된 향수: 0
기준 향수: Encre Noire / atr:103 / fr:1834


,perfume_id,score,perfume_name,brand,gender
0,fr:52108,0.591214,ecorce-d-argent,korloff-paris,unisex
1,fr:21369,0.590164,encre-noire-sport,lalique,men
2,fr:24954,0.523191,elite-gentleman-untailored,avon,men
3,fr:40857,0.478018,radiant-woods,zara,women
4,fr:26226,0.466938,capim-cheiroso,companhia-da-terra,unisex
5,fr:28252,0.450208,rose-royale,lalique,unisex
6,fr:56397,0.443623,vetiver,ajmal,unisex
7,fr:34046,0.442814,yang,o-boticario,men
8,fr:18072,0.436566,infinito,nobile-1942,unisex
9,fr:21854,0.430291,hommage-a-l-homme-voyageur,lalique,men


## 4. 협업 필터링: Item-based KNN

동작 예시에서는 `user_ratings.csv`의 유효 평점 전체를 사용. 평가 시에는 학습용으로 분리한 평점만 입력.

1. 사용자 × 향수 희소 평점 행렬 $R$을 구성.
2. **향수 × 사용자** 벡터 사이의 코사인 유사도를 계산.
3. 공동 평가자가 너무 적은 향수 쌍을 제외하고, 공동 평가자 수에 따라 유사도를 축소.
4. 후보 향수마다 해당 사용자가 이미 평가한 향수 중 유사도가 높은 최대 $k$개를 선택.
5. 이웃의 실제 평점을 유사도로 가중 평균하여 예상 평점을 계산.

기본 설정은 `k=20`, 최소 공동 평가자 `m=2`, 축소 계수 `λ=10`, 최소 예측 이웃 1개. 모두 검증 전 초기 설정값.

원시 평점 코사인을 사용하며 사용자 평균 중심화는 적용하지 않음. 평점이 한 개인 사용자는 평균 중심화하면 해당 평점이 0이 되는 점을 고려.

행렬의 저장되지 않은 칸은 수학적 계산에서 0으로 표현되며, KNN에는 관측된 1~10점만 사용. 코사인의 내적은 공동 평가에서, 각 노름은 해당 향수의 전체 관측 평점에서 계산.

**이웃이 부족한 경우:** 알려진 사용자면 학습 이력의 사용자 평균, 새 사용자이면 해당 향수의 학습 평균, 사용자·향수 모두 처음이면 학습 전체 평균을 반환. `method`로 이를 표시. 추천 목록은 기본적으로 `method="knn"`인 후보만 포함하며, 후보가 부족하면 10개보다 적게 반환할 수 있음.

In [ ]:
# 향수 간 평점 유사도와 이웃 평점의 가중 평균으로 추천
class ItemKNNRecommender(BaseEstimator):

    # 이웃 수, 공동 평가자 기준, 유사도 보정값을 설정
    def __init__(self, k=20, min_common_users=2, shrinkage=10.0, min_neighbors=1):
        for name, value in [("k", k), ("min_common_users", min_common_users),
                            ("min_neighbors", min_neighbors)]:
            _positive_int(value, name)
        if min_neighbors > k:
            raise ValueError("min_neighbors cannot exceed k")
        if not np.isfinite(shrinkage) or shrinkage < 0:
            raise ValueError("shrinkage must be finite and nonnegative")
        self.k, self.min_common_users = int(k), int(min_common_users)
        self.shrinkage, self.min_neighbors = float(shrinkage), int(min_neighbors)

    # 입력 평점만으로 희소 행렬·향수 유사도·대체 예측용 평균을 계산
    def fit(self, train_ratings, item_metadata=None):
        # 이 메서드에 전달된 평점만 사용하므로 평가 시 학습 데이터만 사용
        ratings = _validate_ratings(train_ratings)
        if ratings.empty:
            raise ValueError("Training ratings must not be empty")
        # 문자열 ID와 행렬 행·열의 대응을 일정하게 유지하기 위한 정렬된 인덱스를 생성
        self.user_ids_ = pd.Index(sorted(ratings.user_id.unique()), name="user_id")
        self.item_ids_ = pd.Index(sorted(ratings.perfume_id.unique()), name="perfume_id")
        users, items = self.user_ids_.get_indexer(ratings.user_id), self.item_ids_.get_indexer(ratings.perfume_id)
        # 사용자 × 향수의 행렬에 실제 관측 평점만 저장
        self.ratings_matrix_ = csr_matrix(
            (ratings.scent_rating.to_numpy(dtype=np.float32), (users, items)),
            shape=(len(self.user_ids_), len(self.item_ids_)))
        # 평점 크기와 별개로 관측 여부는 1
        observed = self.ratings_matrix_.copy().astype(np.int32)
        observed.data[:] = 1
        # 관측 행렬을 곱해 각 향수 쌍을 모두 평가한 사용자 수를 구함
        self.common_users_ = (observed.T @ observed).tocsr()
        # 공동 평가자가 기준보다 적으면 제외하고, 나머지는 n/(n+축소계수)로 보정
        # 보정에 경우 평점 데이터가 희소하여 소수의 평가자만으로 계산된 유사도가 과대평가되는 것을 방지하기 위해 사용했습니다.
        reliability = self.common_users_.copy().astype(np.float32)
        common = reliability.data.copy()
        reliability.data = np.where(common >= self.min_common_users,
                                    common / (common + self.shrinkage), 0)
        reliability.eliminate_zeros()
        # 행렬을 전치해 향수끼리 코사인 유사도를 계산한 뒤 신뢰도 보정을 적용
        self.similarities_ = cosine_similarity(self.ratings_matrix_.T.tocsr(), dense_output=False).multiply(reliability).tocsr()
        # 향수 자기 자신은 이웃으로 사용하지 않음
        self.similarities_.setdiag(0)
        self.similarities_.eliminate_zeros()
        self.similarities_.sort_indices()
        # 이웃이 부족할 때 사용할 평균도 입력받은 학습 평점에서만 계산
        self.user_means_ = ratings.groupby("user_id").scent_rating.mean()
        self.item_means_ = ratings.groupby("perfume_id").scent_rating.mean()
        self.global_mean_ = float(ratings.scent_rating.mean())
        self.metadata_ = pd.DataFrame({"perfume_id": self.item_ids_})

        if item_metadata is not None:
            _require_columns(item_metadata, ["perfume_id"])
            _check_ids(item_metadata, ["perfume_id"])
            metadata = item_metadata.copy()
            metadata["perfume_id"] = metadata.perfume_id.astype(str)
            if metadata.perfume_id.duplicated().any():
                raise ValueError("Duplicate IDs in item_metadata")
            columns = [c for c in ["perfume_id", "perfume_name", "brand", "url"] if c in metadata]
            self.metadata_ = self.metadata_.merge(metadata[columns], on="perfume_id", how="left", validate="one_to_one")
        return self

    # 기준 향수와 유사한 향수 및 공동 평가자 수를 반환
    def similar_items(self, perfume_id, top_n=10):
        check_is_fitted(self, ["similarities_"])
        _positive_int(top_n, "top_n")
        index = self.item_ids_.get_indexer([str(perfume_id)])[0]
        if index < 0:
            raise KeyError(f"Item not present in training data: {perfume_id}")
        # 기준 향수의 유사도 행과 공동 평가자 수를 데이터프레임으로 만들어 상위 top_n개를 반환
        row = self.similarities_.getrow(index)
        result = pd.DataFrame({"perfume_id": self.item_ids_[row.indices], "similarity": row.data,
            "n_common_users": self.common_users_[index, row.indices].toarray().ravel()})
        result = result.sort_values(["similarity", "perfume_id"], ascending=[False, True]).head(top_n)
        return result.merge(self.metadata_, on="perfume_id", validate="one_to_one")

    # 사용자–향수 한 쌍의 예상 평점, 예측 방식, 이웃 수를 반환
    def predict_details(self, user_id, perfume_id):
        check_is_fitted(self, ["similarities_"])
        user_id, perfume_id = str(user_id), str(perfume_id)
        user = self.user_ids_.get_indexer([user_id])[0]
        item = self.item_ids_.get_indexer([perfume_id])[0]
        count = 0
        if user >= 0 and item >= 0:
            # 해당 사용자가 실제 평가한 향수의 평점만 이웃 후보로 사용
            history = self.ratings_matrix_.getrow(user)
            sims = self.similarities_[item, history.indices].toarray().ravel()
            eligible = np.flatnonzero(sims > 0)
            # 이 사용자가 평가한 향수 중 양의 유사도가 높은 최대 k개를 선택
            nearest = eligible[np.argsort(-sims[eligible], kind="stable")[:self.k]]
            count = len(nearest)
            if count >= self.min_neighbors:
                # 유사도 × 실제 평점의 합을 유사도 합으로 나눠 예상 평점을 계산
                prediction = np.dot(sims[nearest], history.data[nearest]) / sims[nearest].sum()
                return {"predicted_rating": float(np.clip(prediction, 1, 10)), "method": "knn", "n_neighbors": count}
        # 이웃이 부족한 경우 사용자 평균 → 아이템 평균 → 전체 평균 순으로 대체
        if user >= 0:
            prediction, method = self.user_means_.loc[user_id], "user_mean"
        elif item >= 0:
            prediction, method = self.item_means_.loc[perfume_id], "item_mean"
        else:
            prediction, method = self.global_mean_, "global_mean"
        return {"predicted_rating": float(prediction), "method": method, "n_neighbors": count}

    # 사용자–향수 한 쌍의 예상 평점만 반환
    def predict(self, user_id, perfume_id):
        return self.predict_details(user_id, perfume_id)["predicted_rating"]

    # 여러 사용자–향수 쌍을 입력 순서대로 예상 평점과 예측 방식을 반환
    def predict_pairs(self, pairs):
        check_is_fitted(self, ["similarities_"])
        _require_columns(pairs, ["user_id", "perfume_id"])
        _check_ids(pairs, ["user_id", "perfume_id"])
        # 입력 순서를 유지하고 예측값과 대체 방식(method)을 나란히 반환
        result = pairs[["user_id", "perfume_id"]].astype(str).reset_index(drop=True)
        details = [self.predict_details(u, i) for u, i in result.itertuples(index=False, name=None)]
        return pd.concat([result, pd.DataFrame(details, columns=["predicted_rating", "method", "n_neighbors"])], axis=1)

    # 사용자가 평가한 향수를 제외하고 예상 평점이 높은 향수를 추천
    def recommend(self, user_id, top_n=10, candidate_ids=None, include_fallback=False):
        check_is_fitted(self, ["similarities_"])
        _positive_int(top_n, "top_n")
        user_id = str(user_id)
        candidates = self.item_ids_ if candidate_ids is None else pd.Index(list(map(str, candidate_ids)))
        if not candidates.isin(self.item_ids_).all() or candidates.has_duplicates:
            raise ValueError("Candidates must be distinct items present in training data")
        user = self.user_ids_.get_indexer([user_id])[0]
        # 사용자가 이미 평가한 향수는 추천 후보에서 제외
        seen = self.item_ids_[self.ratings_matrix_.getrow(user).indices] if user >= 0 else []
        candidates = candidates[~candidates.isin(seen)]
        result = self.predict_pairs(pd.DataFrame({"user_id": user_id, "perfume_id": candidates}))
        # 기본 추천 목록에는 평균값 대체가 아닌 실제 KNN 예측만 포함
        if not include_fallback:
            result = result.loc[result.method.eq("knn")]
        result = result.sort_values(["predicted_rating", "n_neighbors", "perfume_id"],
                                   ascending=[False, False, True]).head(top_n)
        return result.merge(self.metadata_, on="perfume_id", how="left", validate="one_to_one")


In [ ]:
# 데모용 전체 데이터 학습으로 성능 평가 때는 train_ratings로 바꿔서 사용

knn_model = ItemKNNRecommender(
    k=20, min_common_users=2, shrinkage=10.0, min_neighbors=1
).fit(ratings, item_metadata=atrafshan)

print("사용자 × 향수 평점 행렬:", knn_model.ratings_matrix_.shape)
print("저장된 실제 평점:", knn_model.ratings_matrix_.nnz)
print("신뢰도 보정 후 유사 향수가 있는 향수 수:",
      int((knn_model.similarities_.getnnz(axis=1) > 0).sum()))
display(knn_model.similar_items(SEED_ATR_ID, top_n=10).drop(columns="url", errors="ignore"))

사용자 × 향수 평점 행렬: (1747, 1050)
저장된 실제 평점: 5565
신뢰도 보정 후 유사 향수가 있는 향수 수: 488


,perfume_id,similarity,n_common_users,perfume_name,brand
0,atr:4809,0.052322,2,Flash Back,Olfactive Studio
1,atr:1149,0.035161,2,Bvlgari Man,BVLGARI
2,atr:1145,0.029058,2,Champion,DAVIDOFF
3,atr:1922,0.027397,2,Starwalker,MONT BLANC
4,atr:252,0.024292,2,Jasmin Noir EDP,BVLGARI
5,atr:1724,0.017243,2,Mont Blanc Legend,MONT BLANC
6,atr:40,0.016959,2,Dunhill Desire Blue,dunhill
7,atr:539,0.015352,2,Versace Pour Homme,VERSACE
8,atr:636,0.015278,2,Yatagan,CARON
9,atr:143,0.014188,2,Joop! Homme,JOOP


## 5. 사용자별 추천 예시

선호 평점이 많은 사용자 한 명을 선택한 동작 예시.

콘텐츠 기반 점수 `score`: 0~1 범위의 코사인 유사도. 

KNN의 `predicted_rating`: 1~10점 예상 평점.

In [ ]:
# 선호 평점이 많은 사용자 한 명을 동작 예시로 선택
liked_counts = linked_ratings.loc[linked_ratings.scent_rating.ge(7)].groupby("user_id").size()
DEMO_USER_ID = liked_counts.sort_values(ascending=False, kind="stable").index[0]
print("데모 사용자:", DEMO_USER_ID)

# 같은 사용자에 대해 콘텐츠 추천과 협업 추천을 각각 확인
cb_recommendations = content_model.recommend_for_user(
    DEMO_USER_ID, train_linked_ratings=linked_ratings, top_n=10, like_threshold=7
)
cf_recommendations = knn_model.recommend(DEMO_USER_ID, top_n=10)

print("콘텐츠 기반 추천")
display(cb_recommendations.drop(columns="url", errors="ignore"))
print("협업 필터링 추천")
display(cf_recommendations.drop(columns="url", errors="ignore"))

# 알 수 없는 사용자/향수와 이웃 부족 시 대체 예측이 구분되는지 확인
demo_pairs = pd.DataFrame({
    "user_id": [DEMO_USER_ID, "atr:user:unseen", DEMO_USER_ID, "atr:user:unseen"],
    "perfume_id": [SEED_ATR_ID, SEED_ATR_ID, "atr:unseen", "atr:unseen"]
})
display(knn_model.predict_pairs(demo_pairs))


데모 사용자: atr:user:35298
콘텐츠 기반 추천


,perfume_id,score,perfume_name,brand,gender
0,fr:11768,0.515144,darley,parfums-de-marly,men
1,fr:56273,0.498521,sedley,parfums-de-marly,unisex
2,fr:1044,0.456114,prada-amber-pour-homme-prada-man,prada,men
3,fr:473,0.440228,original-santal,creed,unisex
4,fr:51037,0.440129,percival,parfums-de-marly,unisex
5,fr:9801,0.439521,eclat-for-men,oriflame,men
6,fr:78001,0.438573,royal-bleu,orientica-premium,unisex
7,fr:30526,0.434477,meydan,the-spirit-of-dubai,unisex
8,fr:30819,0.433462,sedbury,parfums-de-marly,women
9,fr:78036,0.432771,xclusif-oud-bleu,orientica-premium,unisex


협업 필터링 추천


,user_id,perfume_id,predicted_rating,method,n_neighbors,perfume_name,brand
0,atr:user:35298,atr:37275,10.0,knn,6,"Rosendo Mateu Nº 5 Floral, Amber, Sensual Musk",Rosendo Mateu Olfactive Expressions
1,atr:user:35298,atr:14111,10.0,knn,5,Y Le Parfum,YVES SAINT LAURENT
2,atr:user:35298,atr:1965,10.0,knn,5,The One for Men EDP,DOLCE & GABBANA
3,atr:user:35298,atr:40162,10.0,knn,4,The Library Collection Opus XIV Royal Tobacco,AMOUAGE
4,atr:user:35298,atr:10256,10.0,knn,3,Uomo Salvatore Ferragamo Signature,Salvatore Ferragamo
5,atr:user:35298,atr:1139,10.0,knn,3,Bleu de Chanel EDT,CHANEL
6,atr:user:35298,atr:12586,10.0,knn,3,02 L'Air du Desert Marocain,Tauer Perfumes
7,atr:user:35298,atr:1296,10.0,knn,3,Guerlain Homme Intense EDP,GUERLAIN
8,atr:user:35298,atr:13503,10.0,knn,3,Interlude Black Iris Man,AMOUAGE
9,atr:user:35298,atr:15200,10.0,knn,3,Ombre Leather Parfum,TOM FORD


,user_id,perfume_id,predicted_rating,method,n_neighbors
0,atr:user:35298,atr:103,9.757977,knn,2
1,atr:user:unseen,atr:103,7.750000,item_mean,0
2,atr:user:35298,atr:unseen,9.166667,user_mean,0
3,atr:user:unseen,atr:unseen,7.820845,global_mean,0


## 6. 두 데이터셋의 ID를 맞춰 같은 후보에서 추천하기

자동 연결된 향수 중 KNN 학습과 콘텐츠 모델 양쪽에 존재하는 향수로 **후보 집합을 먼저 제한**하여 수행. 그 뒤 각 모델에서 순위를 계산.

`fr:`와 `atr:` ID는 유지하고 `matched_exact` 행만으로 연결. 전체 카탈로그 추천과 공통 카탈로그 추천은 후보 수가 다르므로 평가에서 확실히 구분해야 함.

In [ ]:
# 두 모델 모두 다룰 수 있고 ID가 연결된 향수만 공통 후보로 선정
common = exact_mapping.loc[
    exact_mapping.atrafshan_id.isin(knn_model.item_ids_)
    & exact_mapping.fragrantica_id.isin(content_model.item_index_)
].copy()
print("공통 추천 후보 향수:", len(common))

# 후보를 먼저 제한한 다음 각 모델의 추천 수행
cb_common = content_model.recommend_for_user(
    DEMO_USER_ID, linked_ratings, top_n=10,
    candidate_ids=common.fragrantica_id
).rename(columns={"perfume_id": "fragrantica_id"}).merge(
    common[["atrafshan_id", "fragrantica_id"]],
    on="fragrantica_id", validate="one_to_one"
)
cf_common = knn_model.recommend(
    DEMO_USER_ID, top_n=10, candidate_ids=common.atrafshan_id
).rename(columns={"perfume_id": "atrafshan_id"}).merge(
    common[["atrafshan_id", "fragrantica_id"]],
    on="atrafshan_id", validate="one_to_one"
)
display(cb_common.drop(columns="url", errors="ignore"))
display(cf_common.drop(columns="url", errors="ignore"))


공통 추천 후보 향수: 682


,fragrantica_id,score,perfume_name,brand,gender,atrafshan_id
0,fr:56273,0.498521,sedley,parfums-de-marly,unisex,atr:12747
1,fr:51037,0.440129,percival,parfums-de-marly,unisex,atr:10505
2,fr:30819,0.433462,sedbury,parfums-de-marly,women,atr:1951
3,fr:15158,0.401493,red-power,ferrari,men,atr:1515
4,fr:31876,0.395671,byerley,parfums-de-marly,men,atr:1908
5,fr:15942,0.392569,vince-camuto-for-men,vince-camuto,men,atr:6829
6,fr:143,0.387697,bvlgari-pour-homme,bvlgari,men,atr:9341
7,fr:84109,0.383370,althair,parfums-de-marly,men,atr:117821
8,fr:68225,0.382754,defy,calvin-klein,men,atr:14841
9,fr:27708,0.377481,tag-him,armaf,men,atr:12173


,user_id,atrafshan_id,predicted_rating,method,n_neighbors,perfume_name,brand,fragrantica_id
0,atr:user:35298,atr:37275,10.0,knn,6,"Rosendo Mateu Nº 5 Floral, Amber, Sensual Musk",Rosendo Mateu Olfactive Expressions,fr:46115
1,atr:user:35298,atr:14111,10.0,knn,5,Y Le Parfum,YVES SAINT LAURENT,fr:64718
2,atr:user:35298,atr:10256,10.0,knn,3,Uomo Salvatore Ferragamo Signature,Salvatore Ferragamo,fr:50797
3,atr:user:35298,atr:12586,10.0,knn,3,02 L'Air du Desert Marocain,Tauer Perfumes,fr:4573
4,atr:user:35298,atr:15200,10.0,knn,3,Ombre Leather Parfum,TOM FORD,fr:68716
5,atr:user:35298,atr:2431,10.0,knn,3,Cedrat Boise,MANCERA,fr:15211
6,atr:user:35298,atr:595,10.0,knn,3,L'Eau Bleue d'Issey Pour Homme,ISSEY MIYAKE,fr:724
7,atr:user:35298,atr:6825,10.0,knn,3,Soleil de Jeddah,STEPHANE HUMBERT LUCAS 777,fr:18306
8,atr:user:35298,atr:1014,10.0,knn,2,F by Ferragamo Black,Salvatore Ferragamo,fr:5002
9,atr:user:35298,atr:11164,10.0,knn,2,Eros Flame,VERSACE,fr:52180


## 7. 평가 연결 시 참고

- 현재 학습·추천 결과는 전체 평점을 사용한 동작 예시. 평가 시 KNN 학습과 콘텐츠 사용자 프로필에는 학습용 평점만 사용하고, 테스트 평점은 제외.
- 콘텐츠의 `score`는 0~1 유사도, KNN의 `predicted_rating`은 1~10점 예상 평점. KNN의 `method`로 이웃 기반 예측과 평균값 대체를 구분.

데이터 분할 방식과 성능 지표는 평가 단계에서 별도로 정함.

## 8. 데이터셋별 역할

| 데이터셋 | 모델 구축·평가에서의 역할 |
|---|---|
| Fragrantica | 콘텐츠 기반 모델의 브랜드·향 특징 구성 |
| Atrafshan | KNN 학습용 개인 평점 및 콘텐츠 모델 평가에 사용할 보류 평점 |

Fragrantica 특징으로 콘텐츠 모델을 구축하고 Atrafshan의 보류 평점으로 테스트하는 설계에 대응. KNN은 Atrafshan 평점을 학습·테스트로 분리해 평가 가능.

`user_ratings_linked.csv`는 Atrafshan 평점에 Fragrantica ID를 연결한 파일. 